In [1]:
import sys
sys.path.insert(0, "..")  # make src/ importable from notebooks/
import sqlite3
import re
from pathlib import Path
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize
import textstat
import pysentiment2 as ps2
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

# Import improved cleaning from src (handles SEC EDGAR + Motley Fool)
from src.sentiment import clean_transcript, chunk_text, compute_paragraph_vader

# Try DirectML (AMD GPU) first, graceful fallback to CPU
try:
    import torch_directml
    DEVICE = torch_directml.device()
except (ImportError, OSError, RuntimeError) as e:
    DEVICE = torch.device('cpu')
    print(f'DirectML unavailable ({type(e).__name__}); falling back to {DEVICE}')
print(f'PyTorch device: {DEVICE}')

DB_PATH = '../data/market.db'
TRANSCRIPTS_DIR = Path('../data/transcripts')
sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline



In [ ]:
conn = sqlite3.connect(DB_PATH)
tx_df = pd.read_sql(
    "SELECT * FROM transcripts WHERE status = 200 ORDER BY ticker, year, quarter",
    conn, parse_dates=['pub_date', 'scrape_time'])
conn.close()
print(f'Transcripts in DB (status=200): {len(tx_df)}')
tx_df['file_exists'] = tx_df['file_path'].apply(lambda p: Path(p).exists())
missing = tx_df[~tx_df['file_exists']]
if len(missing) > 0:
    print(f'WARNING: {len(missing)} DB rows have no local file — dropping:')
    for _, r in missing.iterrows():
        print(f'  {r["ticker"]} {r["quarter"]}{r["year"]}: {r["file_path"]}')
tx_df = tx_df[tx_df['file_exists']].copy()
print(f'Transcripts with verified files: {len(tx_df)}')
if len(tx_df) == 0:
    print('\n⚠️  No transcript files on disk. Run 02_transcripts.ipynb first to scrape them.')
    print('   Continuing with empty results — visualizations will be skipped.')
else:
    print(f'\nTickers: {sorted(tx_df["ticker"].unique())}')
    print(f'Total words: {tx_df["word_count"].sum():,}')
    tx_df[['ticker', 'quarter', 'year', 'word_count']].head(10)

In [ ]:
# Load and clean transcripts using the improved src.sentiment.clean_transcript()
# which handles SEC EDGAR 8-K filings AND Motley Fool transcripts.
tx_texts = {}
for _, row in tx_df.iterrows():
    fp = Path(row['file_path'])
    raw = fp.read_text(encoding='utf-8')
    key = (row['ticker'], row['quarter'], row['year'])
    tx_texts[key] = clean_transcript(raw)
print(f'Loaded and cleaned {len(tx_texts)} transcripts')
if tx_texts:
    sample_key = list(tx_texts.keys())[0]
    print(f'\nSample ({sample_key}): first 300 chars:')
    print(tx_texts[sample_key][:300])
    print(f'... ({len(tx_texts[sample_key].split())} words total)')

In [ ]:
vader = SentimentIntensityAnalyzer()
lm = ps2.LM()
sentiment_rows = []
for (ticker, quarter, year), text in tx_texts.items():
    # Use sentence-chunked VADER (not whole-document) for meaningful scores
    vader_features = compute_paragraph_vader(text, vader, sentences_per_chunk=5)
    tokens = lm.tokenize(text)
    lm_score = lm.get_score(tokens)
    total_lm = lm_score['Positive'] + lm_score['Negative'] + 1
    lm_net = (lm_score['Positive'] - lm_score['Negative']) / total_lm
    sentiment_rows.append({
        'ticker': ticker,
        'quarter': quarter,
        'year': year,
        **vader_features,  # unpacks all 14 VADER paragraph-chunk features
        'lm_positive': lm_score['Positive'],
        'lm_negative': lm_score['Negative'],
        'lm_uncertainty': lm_score.get('Uncertainty', 0),
        'lm_litigious': lm_score.get('Litigious', 0),
        'lm_constraining': lm_score.get('Constraining', 0),
        'lm_strong_modal': lm_score.get('StrongModal', 0),
        'lm_weak_modal': lm_score.get('WeakModal', 0),
        'lm_net': lm_net,
        'lm_pos_ratio': lm_score['Positive'] / total_lm,
        'lm_neg_ratio': lm_score['Negative'] / total_lm,
    })
df_sent = pd.DataFrame(sentiment_rows)
print(f'Computed VADER + LM scores for {len(df_sent)} transcripts')
if len(df_sent) > 0:
    print('\n=== Sentiment summary ===')
    # Show key new features: mean, std, pct_neg (not just whole-doc compound)
    display(df_sent[['ticker', 'quarter', 'year', 'vader_mean', 'vader_std',
                      'vader_pct_neg', 'lm_net']]
            .round(4).head(20))
    print(f'\nVADER mean (chunked): mean={df_sent["vader_mean"].mean():.4f}  '
          f'std={df_sent["vader_mean"].std():.4f}')
    print(f'VADER std (dispersion): mean={df_sent["vader_std"].mean():.4f}')
    print(f'LM net:                mean={df_sent["lm_net"].mean():.4f}  '
          f'std={df_sent["lm_net"].std():.4f}')

In [ ]:
USE_FINBERT = True
if USE_FINBERT and len(tx_texts) > 0:
    print(f'Loading FinBERT on {DEVICE} ...')
    # Load model + tokenizer manually and move to DirectML device
    # (transformers pipeline device= param doesn't support DirectML)
    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
    model.to(DEVICE)
    model.eval()
    # FinBERT label order: {0: 'positive', 1: 'negative', 2: 'neutral'}
    print('FinBERT loaded.')
    finbert_results = []
    INFERENCE_BATCH = 16  # process chunks in sub-batches to avoid OOM
    for (ticker, quarter, year), text in tx_texts.items():
        chunks = chunk_text(text, max_words=256)
        all_positive, all_negative, all_neutral = [], [], []
        for i in range(0, len(chunks), INFERENCE_BATCH):
            batch_chunks = chunks[i:i + INFERENCE_BATCH]
            inputs = tokenizer(
                batch_chunks, return_tensors='pt',
                truncation=True, max_length=512, padding=True
            ).to(DEVICE)
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
            # id2label: 0=positive, 1=negative, 2=neutral
            all_positive.extend(probs[:, 0].tolist())
            all_negative.extend(probs[:, 1].tolist())
            all_neutral.extend(probs[:, 2].tolist())
        n = len(chunks)
        label_probs = {
            'positive': float(np.mean(all_positive)),
            'negative': float(np.mean(all_negative)),
            'neutral': float(np.mean(all_neutral)),
        }
        fb_net = label_probs['positive'] - label_probs['negative']
        fb_label = max(label_probs, key=label_probs.get)
        finbert_results.append({
            'ticker': ticker, 'quarter': quarter, 'year': year,
            'finbert_positive': label_probs['positive'],
            'finbert_negative': label_probs['negative'],
            'finbert_neutral': label_probs['neutral'],
            'finbert_net': fb_net,
            'finbert_label': fb_label,
            'finbert_chunks': n,
        })
    df_fb = pd.DataFrame(finbert_results)
    df_sent = df_sent.merge(df_fb, on=['ticker', 'quarter', 'year'], how='left')
    print(f'FinBERT scored {len(df_fb)} transcripts '
          f'(avg {df_fb["finbert_chunks"].mean():.0f} chunks each)')
    print(f'FinBERT net: mean={df_sent["finbert_net"].mean():.4f}  '
          f'std={df_sent["finbert_net"].std():.4f}')
else:
    print('FinBERT skipped (USE_FINBERT=False or no transcripts).')
    for col in ['finbert_positive', 'finbert_negative', 'finbert_neutral',
                'finbert_net', 'finbert_label', 'finbert_chunks']:
        if col not in df_sent.columns:
            df_sent[col] = np.nan

In [ ]:
readability_rows = []
for (ticker, quarter, year), text in tx_texts.items():
    sentences = sent_tokenize(text)
    words = text.split()
    n_sentences = len(sentences)
    n_words = len(words)
    unique_ratio = len(set(w.lower() for w in words)) / max(n_words, 1)
    avg_sent_len = n_words / max(n_sentences, 1)
    readability_rows.append({
        'ticker': ticker, 'quarter': quarter, 'year': year,
        'n_sentences': n_sentences,
        'n_words': n_words,
        'unique_word_ratio': round(unique_ratio, 4),
        'avg_sentence_length': round(avg_sent_len, 1),
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'smog_index': textstat.smog_index(text),
        'automated_readability': textstat.automated_readability_index(text),
        'dale_chall_score': textstat.dale_chall_readability_score(text),
    })
df_read = pd.DataFrame(readability_rows)
df_sent = df_sent.merge(df_read, on=['ticker', 'quarter', 'year'], how='left')
print(f'Computed readability for {len(df_read)} transcripts')
if len(df_read) > 0:
    print('\n=== Readability summary ===')
    print(f'Flesch Reading Ease:  mean={df_read["flesch_reading_ease"].mean():.1f}  '
          f'(higher = easier)')
    print(f'Flesch-Kincaid Grade: mean={df_read["flesch_kincaid_grade"].mean():.1f}')
    print(f'Gunning Fog:          mean={df_read["gunning_fog"].mean():.1f}')
    print(f'Unique word ratio:    mean={df_read["unique_word_ratio"].mean():.3f}')

In [ ]:
conn = sqlite3.connect(DB_PATH)
returns_df = pd.read_sql(
    "SELECT * FROM returns", conn, parse_dates=['earnings_date'])
conn.close()
print(f'Returns table: {len(returns_df)} events, {returns_df["ticker"].nunique()} tickers')
joined = []
unmatched = 0
for _, tx_row in df_sent.iterrows():
    ticker = tx_row['ticker']
    ticker_returns = returns_df[returns_df['ticker'] == ticker].copy()
    if ticker_returns.empty:
        unmatched += 1
        continue
    matched_row = tx_df[(tx_df['ticker'] == ticker) &
                        (tx_df['quarter'] == tx_row['quarter']) &
                        (tx_df['year'] == tx_row['year'])]
    if matched_row.empty:
        unmatched += 1
        continue
    pub_date = matched_row.iloc[0]['pub_date']
    ticker_returns['day_diff'] = abs(
        (ticker_returns['earnings_date'] - pub_date).dt.days)
    closest = ticker_returns.loc[ticker_returns['day_diff'].idxmin()]
    if closest['day_diff'] > 30:
        unmatched += 1
        continue
    row_dict = {k: v for k, v in tx_row.items()}
    row_dict.update({
        'matched_earnings_date': closest['earnings_date'],
        'day_diff': closest['day_diff'],
        'return_1d': closest['return_1d'],
        'return_30d': closest['return_30d'],
        'return_90d': closest['return_90d'],
        'abnormal_1d': closest['abnormal_1d'],
        'abnormal_30d': closest['abnormal_30d'],
        'abnormal_90d': closest['abnormal_90d'],
        'vix_close': closest['vix_close'],
        'is_covid': closest['is_covid'],
    })
    joined.append(row_dict)
if joined:
    df_merged = pd.DataFrame(joined)
    print(f'Matched {len(df_merged)} transcripts to returns data')
    print(f'Unmatched: {unmatched}')
    print(f'Mean day_diff: {df_merged["day_diff"].mean():.1f} days')
else:
    print(f'No matches found ({unmatched} unmatched). Need more data.')
    df_merged = pd.DataFrame()
df_merged.head()

In [ ]:
if len(df_merged) > 0:
    feature_cols = [
        'ticker', 'quarter', 'year',
        # VADER paragraph-chunk features (14 columns)
        'vader_compound', 'vader_pos', 'vader_neg', 'vader_neu',
        'vader_mean', 'vader_std', 'vader_min', 'vader_max',
        'vader_p10', 'vader_p90', 'vader_pct_neg', 'vader_pct_pos',
        'vader_n_chunks', 'vader_n_paragraphs',
        # Loughran-McDonald features
        'lm_positive', 'lm_negative', 'lm_uncertainty', 'lm_litigious',
        'lm_constraining', 'lm_strong_modal', 'lm_weak_modal',
        'lm_net', 'lm_pos_ratio', 'lm_neg_ratio',
        'finbert_positive', 'finbert_negative', 'finbert_neutral',
        'finbert_net', 'finbert_label', 'finbert_chunks',
        'n_sentences', 'n_words', 'unique_word_ratio', 'avg_sentence_length',
        'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog',
        'smog_index', 'automated_readability', 'dale_chall_score',
        'matched_earnings_date',
        'return_1d', 'return_30d', 'return_90d',
        'abnormal_1d', 'abnormal_30d', 'abnormal_90d',
        'vix_close', 'is_covid',
    ]
    keep_cols = [c for c in feature_cols if c in df_merged.columns]
    df_features = df_merged[keep_cols].copy()
    conn = sqlite3.connect(DB_PATH)
    df_features.to_sql('sentiment_features', conn, if_exists='replace', index=False)
    saved = conn.execute("SELECT COUNT(*) FROM sentiment_features").fetchone()[0]
    conn.close()
    print(f'Saved {saved} rows to sentiment_features table')
    print(f'Columns: {df_features.columns.tolist()}')
else:
    print('No data to persist (df_merged is empty).')


In [ ]:
if len(df_merged) < 3:
    print(f'⚠️  Only {len(df_merged)} matched transcripts — '
          f'skipping visualizations (need ≥ 3 for meaningful plots).')
    print('Run the full-scale fetch in 02_transcripts.ipynb first.')
else:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr_cols = [c for c in ['vader_compound', 'lm_net', 'finbert_net',
                              'abnormal_1d', 'abnormal_30d', 'abnormal_90d',
                              'vix_close', 'flesch_reading_ease',
                              'unique_word_ratio']
                 if c in df_merged.columns and df_merged[c].notna().any()]
    corr = df_merged[corr_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax,
                cbar_kws={'label': 'Pearson r'})
    ax.set_title('Sentiment × Returns correlation matrix', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()
    if 'vader_compound' in df_merged.columns and 'abnormal_1d' in df_merged.columns:
        fig, ax = plt.subplots(figsize=(8, 6))
        colors = df_merged['is_covid'].map({True: '#e74c3c', False: '#3498db'})
        ax.scatter(df_merged['vader_compound'], df_merged['abnormal_1d'],
                   c=colors, alpha=0.7, edgecolors='white', s=80)
        for _, row in df_merged.iterrows():
            ax.annotate(f"{row['ticker']} {row['quarter']}{row['year']}",
                        (row['vader_compound'], row['abnormal_1d']),
                        fontsize=7, alpha=0.7,
                        textcoords='offset points', xytext=(5, 5))
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('VADER Compound Sentiment')
        ax.set_ylabel('Abnormal 1-Day Return (XLK-adjusted)')
        ax.set_title('Earnings Call Sentiment vs Next-Day Abnormal Return')
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='#3498db', label='Normal'),
                           Patch(facecolor='#e74c3c', label='COVID window')]
        ax.legend(handles=legend_elements, loc='best')
        plt.tight_layout()
        plt.show()
    if 'vader_compound' in df_merged.columns:
        ticker_sent = df_merged.groupby('ticker').agg(
            vader_mean=('vader_compound', 'mean'),
            vader_std=('vader_compound', 'std'),
            count=('vader_compound', 'count'),
            abn_1d_mean=('abnormal_1d', 'mean'),
        ).sort_values('vader_mean')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        bars = ax1.barh(ticker_sent.index, ticker_sent['vader_mean'],
                        xerr=ticker_sent['vader_std'], capsize=3,
                        color='steelblue', alpha=0.8)
        ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax1.set_xlabel('Mean VADER Compound')
        ax1.set_title('Sentiment by Ticker')
        ax2.barh(ticker_sent.index, ticker_sent['abn_1d_mean'],
                 color='coral', alpha=0.8)
        ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax2.set_xlabel('Mean Abnormal 1d Return')
        ax2.set_title('Abnormal 1d Return by Ticker')
        plt.tight_layout()
        plt.show()
    if 'vader_compound' in df_merged.columns and 'flesch_reading_ease' in df_merged.columns:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(df_merged['flesch_reading_ease'], df_merged['vader_compound'],
                   c='steelblue', alpha=0.7, s=80, edgecolors='white')
        ax.set_xlabel('Flesch Reading Ease (higher = easier)')
        ax.set_ylabel('VADER Compound Sentiment')
        ax.set_title('Sentiment vs Readability')
        plt.tight_layout()
        plt.show()

## Summary & next steps

This notebook computed:
- **VADER** compound sentiment scores
- **Loughran-McDonald** financial dictionary counts (positive, negative,
  uncertainty, litigious, modal words)
- **FinBERT** transformer-based sentiment (if enabled; accelerated via DirectML on AMD GPU)
- **Readability** metrics (Flesch-Kincaid, Gunning Fog, etc.)
- **Linguistic** features (unique word ratio, avg sentence length)

Results were matched to XLK-adjusted abnormal returns and saved to the
`sentiment_features` table in `market.db`.

**If you have < 30 matched transcripts:** the correlation analysis is
underpowered. Run `02_transcripts.ipynb` cell 8 to fetch all ~360 transcripts
first, then re-run this notebook.

**Next notebook ideas:**
- `04_modeling.ipynb` — LightGBM regression predicting abnormal_30d from
  sentiment + VIX + readability features, with SHAP explainability
- `05_portfolio.ipynb` — backtest a long/short strategy based on sentiment
  surprise (actual sentiment vs expected by VIX regime)